In [1]:
#| include: false
# Disable tqdm progress bars; this is a hidden cell on the docs page
import os

os.environ["TQDM_DISABLE"] = "1"

# See the mix of regions around a point

To see which regions lie within a distance of a point, look up the region at every voxel position inside a sphere around it. The share of positions in each region is the share of the sphere in that region. Coordinates are in metres relative to bregma, ordered ML, AP, DV; see [Coordinates and volumes](../explanation/coordinates-and-volumes.qmd).

In [2]:
import numpy as np
import pandas as pd

from iblatlas.atlas import AllenAtlas

ba = AllenAtlas()
point = np.array([-0.0024, -0.002, -0.0006])  # ML, AP, DV in metres from bregma
radius, step = 100e-6, 25e-6                  # metres; step is the atlas voxel size

# Offsets on a grid of whole voxels, kept if they lie inside the sphere
n = round(radius / step)
k = np.arange(-n, n + 1)
offsets = np.stack(np.meshgrid(k, k, k), axis=-1).reshape(-1, 3)
offsets = offsets[(offsets ** 2).sum(axis=1) <= n ** 2] * step
print(len(offsets), 'voxel positions in the sphere')

ids = ba.get_labels(point + offsets)
pd.Series(ba.regions.id2acronym(ids)).value_counts(normalize=True)

257 voxel positions in the sphere


VISa4      0.731518
VISa2/3    0.151751
VISa5      0.116732
Name: proportion, dtype: float64

The point is in VISa4, layer 4 of the Anterior area VISa, and part of the sphere reaches the layers above and below it. The grid step is the voxel size, so the shares are a sample at 25 µm spacing.

## A coarser mapping

With `mapping='Beryl'` the cortical layers are grouped into their area, so the whole sphere is in VISa. See [Brain region mappings](../explanation/brain-region-mappings.qmd).

In [3]:
ids = ba.get_labels(point + offsets, mapping='Beryl')
pd.Series(ba.regions.id2acronym(ids)).value_counts(normalize=True)

VISa    1.0
Name: proportion, dtype: float64

To look up the region at the point alone, see [Find which brain region a coordinate is in](find-the-region-of-a-point.ipynb).